# ScanNet HHA Phase 0 — Convention Validation + Drop List

**Purpose.** Before any HHA preprocessing run, validate that the math composition
`R_scannet = orthogonalize(axisAlignment[:3,:3]) @ pose[:3,:3]` gives a world frame where +Z is gravity-up,
and produce a drop list of broken / missing-axis-alignment / drifted-pose / high-NaN scenes.

The output JSON lands at `/content/drive/MyDrive/datasets/scannet_drop_list.json` and is read by
`notebooks/scannet_preprocess.ipynb`. Re-run this notebook whenever ScanNet's raw data updates or
when you want to update the drop-list thresholds.

## Two-pass probe

- **Pass A (5 scenes, deep dive)**: T0 camera convention, T1 orthogonality residuals,
  T2 composition direction, T4 per-scene pose-drift distribution, T5 depthShift values.
- **Pass B (200 scenes, shallow)**: relative-floor test (`floor_z relative to scene z-range`),
  per-scene NaN-rate, drift-frame fraction. Builds the drop list.

Output schema (see `src/data_utils/hha/scannet_intrinsics.py:load_drop_list`):
```json
{
  "convention_verified": true,
  "axis_alignment_inverted": false,
  "scenes": {"scene0023_01": "broken_axisAlignment_relative_floor=0.74", ...}
}
```

In [ ]:
# === Mount Drive + clone repo + install deps ===
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_URL = 'https://github.com/<your-user>/Multi-Stream-Neural-Networks.git'  # TODO: set
REPO_DIR = '/content/Multi-Stream-Neural-Networks'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# === Configure paths ===
SCANNET_RAW_ROOT = '/content/drive/MyDrive/datasets/scannet_raw'  # TODO: set to your raw ScanNet dir on Drive
DROP_LIST_OUT = '/content/drive/MyDrive/datasets/scannet_drop_list.json'

# Raw ScanNet layout per scene:
#   <SCANNET_RAW_ROOT>/<scene_id>/
#       <scene_id>.txt         (axisAlignment + sensorType + ...)
#       <scene_id>.sens        (sensor stream — pose, depth, intrinsics)
# We don't extract from .sens here for Phase 0; we only need axisAlignment +
# pose + K + a few depth frames per scene. Phase 3's full preprocess does
# the heavy .sens extraction.

# Phase 0 sample sizes (deep + shallow).
PASS_A_N = 5
PASS_B_N = 200
SEED = 42


In [ ]:
# === Repo-side imports ===
import json
import numpy as np
import torch

from src.data_utils.hha import (
    compute_hha,
    read_intrinsic_depth,
    read_axis_alignment,
    read_pose,
    angular_distance_deg,
)
from src.data_utils.hha.scannet_intrinsics import _orthogonalize, ScannetSceneMeta


In [ ]:
# === Discover scenes on Drive ===
import os
all_scenes = sorted([
    s for s in os.listdir(SCANNET_RAW_ROOT)
    if os.path.isdir(os.path.join(SCANNET_RAW_ROOT, s)) and s.startswith('scene')
])
print(f'Found {len(all_scenes)} scenes')

rng = np.random.default_rng(SEED)
pass_a_scenes = list(rng.choice(all_scenes, size=min(PASS_A_N, len(all_scenes)), replace=False))
pass_b_scenes = list(rng.choice(all_scenes, size=min(PASS_B_N, len(all_scenes)), replace=False))
print(f'Pass A scenes: {pass_a_scenes}')
print(f'Pass B scenes: {len(pass_b_scenes)}')


In [ ]:
# === Pass A: T1 orthogonality residual, T2 composition direction, T4 drift, T5 depthShift ===
# This pass requires the .sens extracted intrinsic_depth.txt + pose/<i>.txt files.
# If you haven't extracted them yet, you can do so here with the ScanNet SensorData.py
# or just point at an already-extracted dir. For brevity we assume the per-scene dir
# has been extracted to <SCANNET_RAW_ROOT>/<scene>/ with intrinsic/, pose/, depth/.

t1_residuals = []
t5_depth_shifts = []
t2_composition_results = []  # 'canonical' or 'inverted' per scene
t4_drift_distributions = {}

for scene_id in pass_a_scenes:
    scene_dir = os.path.join(SCANNET_RAW_ROOT, scene_id)
    print(f'\n--- Scene {scene_id} ---')

    # T1: orthogonality residual
    raw_align = read_axis_alignment(scene_dir, inverted=False)
    if raw_align is None:
        print(f'  axisAlignment missing -> fall-back to identity')
        continue
    M = raw_align[:3, :3]
    R_clean = _orthogonalize(M)
    residual = float(np.linalg.norm(M - R_clean))
    t1_residuals.append(residual)
    print(f'  T1 orthogonality residual: {residual:.4f} (large => scale folded in)')

    # T5: depthShift from <scene>.txt
    depth_shift = 1000.0  # default; parse if non-standard
    txt = os.path.join(scene_dir, f'{scene_id}.txt')
    if os.path.isfile(txt):
        with open(txt) as f:
            for line in f:
                if line.startswith('depthShift'):
                    depth_shift = float(line.split('=', 1)[1].strip())
                    break
    t5_depth_shifts.append(depth_shift)
    print(f'  T5 depthShift: {depth_shift}')

    # Sample a frame to test T2 composition direction.
    pose = read_pose(scene_dir, 0)
    if pose is None:
        print(f'  pose 0 invalid; skipping T2')
        continue
    K = read_intrinsic_depth(scene_dir)
    R_canonical = R_clean @ pose[:3, :3]

    # Try canonical first: if the relative-floor test passes, mark canonical.
    # Load depth frame 0 (assumes already extracted to <scene>/depth/0.png).
    from PIL import Image
    depth_path = os.path.join(scene_dir, 'depth', '0.png')
    if not os.path.isfile(depth_path):
        print(f'  depth 0 missing; cannot test T2'); continue
    depth_mm = np.asarray(Image.open(depth_path), dtype=np.uint16)
    depth_m = depth_mm.astype(np.float32) / depth_shift

    valid = depth_m > 0
    if valid.sum() == 0:
        print(f'  depth 0 all zero; skipping'); continue
    H, W = depth_m.shape
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    x = np.arange(1, W+1, dtype=np.float64)
    y = np.arange(1, H+1, dtype=np.float64)
    xx, yy = np.meshgrid(x, y)
    pts_cam = np.stack([(xx-cx)*depth_m/fx, (yy-cy)*depth_m/fy, depth_m], axis=-1)

    def relative_floor(R):
        pw = pts_cam @ R.T
        z = pw[..., 2][valid]
        if z.size < 100: return None
        z_lo, z_hi = np.percentile(z, [1, 99])
        z_range = max(z_hi - z_lo, 1e-3)
        floor_z = np.percentile(z, 5)
        return (floor_z - z_lo) / z_range

    rf_canonical = relative_floor(R_canonical)
    rf_inverted = relative_floor(R_clean.T @ pose[:3, :3])
    is_canonical = rf_canonical is not None and rf_canonical < 0.10
    is_inverted = rf_inverted is not None and rf_inverted < 0.10
    if is_canonical and not is_inverted:
        t2_composition_results.append('canonical')
        print(f'  T2: canonical (rf_canonical={rf_canonical:.3f})')
    elif is_inverted and not is_canonical:
        t2_composition_results.append('inverted')
        print(f'  T2: INVERTED (rf_inverted={rf_inverted:.3f})')
    else:
        t2_composition_results.append('ambiguous')
        print(f'  T2: ambiguous (rf_can={rf_canonical}, rf_inv={rf_inverted})')

print(f'\nT1 residuals: min={min(t1_residuals):.4f}, max={max(t1_residuals):.4f}')
print(f'T5 depthShifts: {t5_depth_shifts} (expect all 1000)')
print(f'T2 verdicts: {t2_composition_results}')
verdict = 'canonical' if t2_composition_results.count('canonical') >= len(t2_composition_results) - 1 else 'inverted'
print(f'==> axis_alignment_inverted = {verdict == "inverted"}')
AXIS_ALIGNMENT_INVERTED = (verdict == 'inverted')


In [ ]:
# === Pass B: relative-floor test + NaN rate + pose drift across 200 scenes ===
from PIL import Image

drop_entries = {}
nan_rates = []
drift_fractions = []

for scene_id in pass_b_scenes:
    scene_dir = os.path.join(SCANNET_RAW_ROOT, scene_id)

    raw_align = read_axis_alignment(scene_dir, inverted=AXIS_ALIGNMENT_INVERTED)
    if raw_align is None:
        drop_entries[scene_id] = 'missing_axisAlignment'
        # not auto-dropped (fall back to identity at preprocess time)
        continue

    R_align = _orthogonalize(raw_align[:3, :3])

    # Sample up to 5 frames per scene for the drift + relative-floor check.
    pose_dir = os.path.join(scene_dir, 'pose')
    depth_dir = os.path.join(scene_dir, 'depth')
    if not (os.path.isdir(pose_dir) and os.path.isdir(depth_dir)):
        continue
    pose_files = sorted(f for f in os.listdir(pose_dir) if f.endswith('.txt'))[:50]
    if not pose_files:
        continue

    sample_idxs = list(range(0, len(pose_files), max(1, len(pose_files)//5)))[:5]
    R_list = []
    for fi in sample_idxs:
        pose = read_pose(scene_dir, int(pose_files[fi].split('.')[0]))
        if pose is None: continue
        R_list.append(R_align @ pose[:3, :3])
    if len(R_list) < 2:
        continue

    # T4: drift distribution; reference = median frame's R
    ref_R = R_list[len(R_list) // 2]
    drifts = [angular_distance_deg(R, ref_R) for R in R_list]
    drift_frac = sum(d > 15.0 for d in drifts) / len(drifts)
    drift_fractions.append(drift_frac)

    # T3: relative-floor test on the median frame
    fi = sample_idxs[len(sample_idxs)//2]
    depth_path = os.path.join(depth_dir, pose_files[fi].replace('.txt', '.png'))
    if not os.path.isfile(depth_path):
        continue
    depth_m = np.asarray(Image.open(depth_path), dtype=np.uint16).astype(np.float32) / 1000.0
    valid = depth_m > 0
    nan_rate = 1.0 - valid.mean()
    nan_rates.append(nan_rate)
    if nan_rate > 0.30:
        drop_entries[scene_id] = f'high_nan_rate={nan_rate:.2f}'
        continue
    if drift_frac > 0.10:
        drop_entries[scene_id] = f'pose_drift_{int(drift_frac*100)}pct_frames>15deg'
        continue

    K = read_intrinsic_depth(scene_dir)
    H, W = depth_m.shape
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    x = np.arange(1, W+1, dtype=np.float64); y = np.arange(1, H+1, dtype=np.float64)
    xx, yy = np.meshgrid(x, y)
    pts_cam = np.stack([(xx-cx)*depth_m/fx, (yy-cy)*depth_m/fy, depth_m], axis=-1)
    pose = read_pose(scene_dir, int(pose_files[fi].split('.')[0]))
    R_scannet = R_align @ pose[:3, :3]
    pw = pts_cam @ R_scannet.T
    z = pw[..., 2][valid]
    if z.size < 100: continue
    z_lo, z_hi = np.percentile(z, [1, 99])
    z_range = max(z_hi - z_lo, 1e-3)
    floor_z = np.percentile(z, 5)
    rf = (floor_z - z_lo) / z_range
    if rf > 0.50:
        drop_entries[scene_id] = f'broken_axisAlignment_relative_floor={rf:.2f}'

print(f'\nPass B summary:')
print(f'  Scenes scanned: {len(pass_b_scenes)}')
print(f'  Dropped: {len(drop_entries)}')
print(f'  Mean NaN rate: {np.mean(nan_rates):.3f}')
print(f'  Mean drift fraction: {np.mean(drift_fractions):.3f}')


In [ ]:
# === Save drop list to Drive ===
out = {
    'convention_verified': True,
    'axis_alignment_inverted': bool(AXIS_ALIGNMENT_INVERTED),
    'scenes': drop_entries,
    'phase0_metadata': {
        'pass_a_n': PASS_A_N,
        'pass_b_n': PASS_B_N,
        'seed': SEED,
        't1_residual_max': float(max(t1_residuals)) if t1_residuals else None,
        't5_unique_depth_shifts': sorted(set(t5_depth_shifts)),
        'mean_nan_rate': float(np.mean(nan_rates)) if nan_rates else None,
        'mean_drift_fraction': float(np.mean(drift_fractions)) if drift_fractions else None,
    },
}
os.makedirs(os.path.dirname(DROP_LIST_OUT), exist_ok=True)
with open(DROP_LIST_OUT, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Wrote {DROP_LIST_OUT}')
print(f'Total scenes flagged: {len(drop_entries)}')
print(f'  Auto-dropped (broken/high_nan/drift): {sum(1 for v in drop_entries.values() if not v.startswith("missing_"))}')
print(f'  Soft-flagged (missing_axisAlignment, fall back to identity): {sum(1 for v in drop_entries.values() if v.startswith("missing_"))}')
